# Testing AST and CFG

In [2]:
import ast
from py2cfg import CFGBuilder

In [3]:
code_without_break = """
def loop_without_break(x):
    while x > 0:
        x += 1
"""

code_with_break = """
def loop_with_break(x):
    while x > 0:
        if x > 10:
            break
        x += 1
"""

In [4]:
# --- Hilfsfunktion zum Anzeigen des AST ---
def print_ast(code_str):
    tree = ast.parse(code_str)
    print(ast.dump(tree, indent=4))

print("=== AST ohne break ===")
print_ast(code_without_break)
print("\n=== AST mit break ===")
print_ast(code_with_break)

=== AST ohne break ===
Module(
    body=[
        FunctionDef(
            name='loop_without_break',
            args=arguments(
                args=[
                    arg(arg='x')]),
            body=[
                While(
                    test=Compare(
                        left=Name(id='x', ctx=Load()),
                        ops=[
                            Gt()],
                        comparators=[
                            Constant(value=0)]),
                    body=[
                        AugAssign(
                            target=Name(id='x', ctx=Store()),
                            op=Add(),
                            value=Constant(value=1))])])])

=== AST mit break ===
Module(
    body=[
        FunctionDef(
            name='loop_with_break',
            args=arguments(
                args=[
                    arg(arg='x')]),
            body=[
                While(
                    test=Compare(
                        left=Name(id='x', ctx

In [9]:
def is_terminating(func_ast: ast.FunctionDef) -> bool:

    def check_node(node):
        if isinstance(node, ast.While):
            # while True ohne break → Endlosschleife
            if isinstance(node.test, ast.Constant) and node.test.value is True:
                if not any(isinstance(n, ast.Break) for n in ast.walk(node)):
                    return False
            # Unterknoten prüfen
            for child in node.body:
                if not check_node(child):
                    return False
        # Für If / andere Kontrollstrukturen ebenfalls Unterknoten prüfen
        for child in getattr(node, 'body', []):
            if not check_node(child):
                return False
        for child in getattr(node, 'orelse', []):
            if not check_node(child):
                return False
        return True

    return check_node(func_ast)

In [11]:
# Beispielnutzung:
tree_with_break = ast.parse("""
def loop_with_break(x):
    while x > 0:
        if x > 10:
            break
        x += 1
""")

tree_without_break = ast.parse("""
def loop_without_break(x):
    while True:
        x += 1
""")

print(is_terminating(tree_with_break.body[0]))    # True
print(is_terminating(tree_without_break.body[0])) # False

True
False


In [12]:
print(tree_with_break.body[0])

FunctionDef(name='loop_with_break', args=arguments(posonlyargs=[], args=[arg(arg='x', annotation=None, type_comment=None)], vararg=None, kwonlyargs=[], kw_defaults=[], kwarg=None, defaults=[]), body=[While(test=Compare(left=Name(...), ops=[Gt(...)], comparators=[Constant(...)]), body=[If(test=Compare(...), body=[Break(...)], orelse=[]), AugAssign(target=Name(...), op=Add(...), value=Constant(...))], orelse=[])], decorator_list=[], returns=None, type_comment=None, type_params=[])
